Code to create vapour pressure deficit (VPD) projections from daily tasmax and hursmin projections.

Equations saturation vapour pressure http://www.bom.gov.au/climate/how/newproducts/images/IDCJHC02_notes.txt

Vapour pressure = exp (1.8096 + (17.269425 * Dew_Point)/(237.3 + Dew_Point))

Saturated Vapour pressure = exp (1.8096 + (17.269425 * Air_Temperature)/(237.3 + Air_Temperature))

Relative Humidity = Vapour pressure / Saturated vapour pressure * 100

Rearrange the formulae to get:

Vapour pressure = rh * 0.0061094 * exp((17.652 * t)/(243.04 + t))

A nice explainer about the relevance of VPD to fire: https://blog.ucsusa.org/carly-phillips/what-is-vapor-pressure-deficit-vpd-and-what-is-its-connection-to-wildfires/

In [1]:
import dask
from dask.distributed import Client, wait
from dask import delayed

client = Client()

client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/8787/status,
Dashboard: /proxy/8787/status,Workers: 7
Total threads: 14,Total memory: 63.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:38085,Workers: 7
Dashboard: /proxy/8787/status,Total threads: 14
Started: Just now,Total memory: 63.00 GiB
Comm: tcp://127.0.0.1:44557,Total threads: 2
Dashboard: /proxy/32795/status,Memory: 9.00 GiB
Nanny: tcp://127.0.0.1:35961,


2025-04-10 13:59:19,163 - distributed.semaphore - WARNING - Tried to release Lock or Semaphore but it was already released: name='/g/data/ia39/ncra/bushfire/vpd/NorESM2-MM/ssp370/r1i1p1f1/BARPA-R/v1-r1/day/ssp370_NorESM2-MM_BARPA-R_gwl1.2_vpd.nc', lease_id='a9808c2414a441279fefd85da105ec13'. This can happen if the Lock or Semaphore timed out before.
2025-04-10 13:59:48,945 - distributed.semaphore - WARNING - Tried to release Lock or Semaphore but it was already released: name='/g/data/ia39/ncra/bushfire/vpd/NorESM2-MM/ssp370/r1i1p1f1/BARPA-R/v1-r1/day/ssp370_NorESM2-MM_BARPA-R_gwl1.2_vpd.nc', lease_id='73ae6b8bb0344cf59e5745454cca9f96'. This can happen if the Lock or Semaphore timed out before.
2025-04-10 14:00:52,491 - distributed.semaphore - WARNING - Tried to release Lock or Semaphore but it was already released: name='/g/data/ia39/ncra/bushfire/vpd/NorESM2-MM/ssp370/r1i1p1f1/BARPA-R/v1-r1/day/ssp370_NorESM2-MM_BARPA-R_gwl1.2_vpd.nc', lease_id='fc39972d21b846ac881867d88e8a1e96'. Thi

In [2]:
#import all the stuff
import xarray as xr
import numpy as np
import pandas as pd
from datetime import timedelta
import glob
import sys
sys.path.append("/g/data/mn51/users/nb6195/project/gwls/")
import gwl

In [3]:
#function to compute VPD from tasmax and rh datasets
#input: are datasets of rh and tasmax
#output: datasets of vpd and monthly_mean_vpd

def vpd_calc(ds_rh, ds_tasmax):
#    vpd = (1 - ds_rh/100) * 0.61094 * np.exp((17.652 * ds_tasmax)/(243.04 + ds_tasmax)) #originally used funtion
    vpd = (1 - ds_rh/100) * 6.1094 * np.exp ((17.625 * ds_tasmax)/(243.04 + ds_tasmax)) #the formula that Blair uses from http://www.bom.gov.au/research/publications/cawcrreports/CTR_024.pdf
    monthly_mean_vpd = vpd.groupby('time.month').mean('time', keep_attrs=True)
    
    vpd.attrs = {
        'long_name': 'Daily maximum vapour dressure deficit computed from tasmax and hursmin',
        'standard_name': 'vpd',
        'units': 'hPa',
#        'regrid_method': 'bilinear'
    }
    ds_vpd = xr.Dataset({'vpd' : vpd})

    monthly_mean_vpd.attrs = {
        'long_name': 'Monthly mean vapour dressure deficit computed from tasmax and hursmin',
        'standard_name': 'monthly_mean_vpd',
        'units': 'hPa',
    }
    ds_monthly_mean_vpd = xr.Dataset({'monthly_mean_vpd' : monthly_mean_vpd})
    return ds_vpd, ds_monthly_mean_vpd

BOM to do:

- NorESM2-MM

BOM done: 
- ACCESS-CM2 (r4i1p1f1)
- ACCESS-ESM1-5 (r6i1p1f1)
- EC-Earth3 (r1i1p1f1)
- MPI-ESM1-2-HR (r1i1p1f1)
- CESM2
- CMCC-ESM2

CSIRO done: 
- ACCESS-CM2 (r4i1p1f1)
- ACCESS-ESM1-5 (r6i1p1f1)
- CNRM-ESM2-1 (r1i1p1f2)
- CESM2
- CMCC-ESM2
- EC-Earth3
- NorESM2-MM

In [104]:
#Set parameters
CMIP='CMIP6'
#AGENCY = 'CSIRO' 
#RCM = 'CCAM-v2203-SN'
AGENCY = 'BOM' 
RCM = 'BARPA-R'

#GCM = 'ACCESS-ESM1-5' #ensemble = 'r6i1p1f1' #Done
#GCM = 'ACCESS-CM2' #ensemble = 'r4i1p1f1' #Done
#GCM = 'CNRM-ESM2-1' #ensemble = 'r1i1p1f2' #CSIRO Done, no BOM
#GCM = 'MPI-ESM1-2-HR' #ensemble = 'r1i1p1f1' #BOM done, no CSIRO
#GCM = 'CESM2' #ensemble = 'r11i1p1f1' #Done
#GCM = 'CMCC-ESM2' #ensemble = 'r1i1p1f1' #Done
#GCM = 'EC-Earth3' #ensemble = 'r1i1p1f1' #Done

GCM = 'NorESM2-MM' #datetime issue
ensemble = 'r1i1p1f1'

#pathway = 'ssp126'
pathway = 'ssp370'

output_dir = '/g/data/ia39/ncra/bushfire/vpd/'
output_dir_mm = '/g/data/ia39/ncra/bushfire/vpd/monthly_mean/'

In [105]:
#read in RCM files
var1 = 'tasmax'

#ddir = f"/g/data/ia39/australian-climate-service/release/CORDEX/output-Adjust/{CMIP}/bias-adjusted-input/AUST-05i/{AGENCY}/{GCM}/{pathway}/{ensemble}/{RCM}/v1-r1/day/{var1}/v20241216"
#infiles1=glob.glob(ddir+f'/{var1}_AUST-05i_{GCM}_{pathway}_{ensemble}_{AGENCY}_{RCM}_v1-r1_day_*.nc')
#tasmax_master_ds = xr.open_mfdataset(infiles1)

ddir = f"/g/data/kj66/CORDEX/output/{CMIP}/DD/AUST-05i/{AGENCY}/{GCM}/{pathway}/{ensemble}/{RCM}/v1-r1/day/{var1}/v20241216"
infiles1=glob.glob(ddir+f'/{var1}_AUST-05i_{GCM}_{pathway}_{ensemble}_{AGENCY}_{RCM}_v1-r1_day_*.nc')
tasmax_master_ds = xr.open_mfdataset(infiles1)

In [106]:
var2 = 'hursmin'

#ddir = f"/g/data/ia39/australian-climate-service/release/CORDEX/output-Adjust/{CMIP}/bias-adjusted-input/AUST-05i/{AGENCY}/{GCM}/{pathway}/{ensemble}/{RCM}/v1-r1/day/{var2}/v20241216"
#infiles2=glob.glob(ddir+f'/{var2}_AUST-05i_{GCM}_{pathway}_{ensemble}_{AGENCY}_{RCM}_v1-r1_day_*.nc')
#hursmin_master_ds = xr.open_mfdataset(infiles2)

#hursmin_AUST-05i_CESM2_ssp370_r11i1p1f1_BOM_BARPA-R_v1-r1_day_20760101-20761231.nc

ddir2 = f"/g/data/kj66/CORDEX/output/{CMIP}/DD/AUST-05i/{AGENCY}/{GCM}/{pathway}/{ensemble}/{RCM}/v1-r1/day/{var2}/v20241216"
infiles2=glob.glob(ddir2+f'/{var2}_AUST-05i_{GCM}_{pathway}_{ensemble}_{AGENCY}_{RCM}_v1-r1_day_*.nc')
hursmin_master_ds = xr.open_mfdataset(infiles2)

Need to check that the time of tasmax and the time of hursmin align, else a shift in the time will be required for the calculation to work - uncomment code 2 cells below if the shift is needed

In [107]:
tasmax_master_ds

<xarray.Dataset> Size: 78GB
Dimensions:    (time: 31390, lat: 691, lon: 886, bnds: 2)
Coordinates:
  * time       (time) object 251kB 2015-01-01 12:00:00 ... 2100-12-31 12:00:00
  * lat        (lat) float64 6kB -44.5 -44.45 -44.4 ... -10.1 -10.05 -10.0
  * lon        (lon) float64 7kB 112.0 112.0 112.1 112.2 ... 156.2 156.2 156.2
Dimensions without coordinates: bnds
Data variables:
    tasmax     (time, lat, lon) float32 77GB dask.array<chunksize=(1, 691, 886), meta=np.ndarray>
    lat_bnds   (time, lat, bnds) float64 347MB dask.array<chunksize=(365, 691, 2), meta=np.ndarray>
    lon_bnds   (time, lon, bnds) float64 445MB dask.array<chunksize=(365, 886, 2), meta=np.ndarray>
    time_bnds  (time, bnds) object 502kB dask.array<chunksize=(365, 2), meta=np.ndarray>
Attributes: (12/45)
    Conventions:               CF-1.10, ACDD-1.3
    title:                     CORDEX-CMIP6-based regridded and calibrated da...
    source:                    ssp370 run using NCC NorESM2-MM r1i1p1f1. Forc...
    source_type:               ARCM
    project:                   BARPA
    program:                   Australian Climate Service (ACS)
    ...                        ...
    history:                   Wed Jun 19 10:34:11 2024: /g/data/xv83/dbi599/...
    references:                https://doi.org/10.25914/z1x6-dq28
    license:                   CC BY 4.0
    acknowledgement:           The production of BARPA was supported with fun...
    input_tracking_id:         e6b7f707-3446-4718-9289-8beec1f16e4e
    grid:                      latitude-longitude with 0.05 degree grid spaci...

In [108]:
hursmin_master_ds

<xarray.Dataset> Size: 78GB
Dimensions:    (time: 31390, lat: 691, lon: 886, bnds: 2)
Coordinates:
  * time       (time) object 251kB 2015-01-01 12:00:00 ... 2100-12-31 12:00:00
  * lat        (lat) float64 6kB -44.5 -44.45 -44.4 ... -10.1 -10.05 -10.0
  * lon        (lon) float64 7kB 112.0 112.0 112.1 112.2 ... 156.2 156.2 156.2
Dimensions without coordinates: bnds
Data variables:
    hursmin    (time, lat, lon) float32 77GB dask.array<chunksize=(1, 691, 886), meta=np.ndarray>
    lat_bnds   (time, lat, bnds) float64 347MB dask.array<chunksize=(365, 691, 2), meta=np.ndarray>
    lon_bnds   (time, lon, bnds) float64 445MB dask.array<chunksize=(365, 886, 2), meta=np.ndarray>
    time_bnds  (time, bnds) object 502kB dask.array<chunksize=(365, 2), meta=np.ndarray>
Attributes: (12/45)
    Conventions:               CF-1.10, ACDD-1.3
    title:                     CORDEX-CMIP6-based regridded and calibrated da...
    source:                    Climate change run using NCC-NorESM2-MM ssp370...
    source_id:                 BARPA-R
    source_type:               ARCM
    project:                   BARPA
    ...                        ...
    history:                   Wed Jun 19 10:37:13 2024: /g/data/xv83/dbi599/...
    references:                https://doi.org/10.25914/z1x6-dq28
    license:                   CC BY 4.0
    acknowledgement:           The production of BARPA was supported with fun...
    input_tracking_id:         2bbdc6e5-8745-4c60-9374-5f90162ca06d
    grid:                      latitude-longitude with 0.05 degree grid spaci...

In [148]:
#Extract time period corresponding to the chosen GWL for tasmax and rh
chosen_gwl = '3.0'

gwl_tasmax = gwl.get_GWL_timeslice(tasmax_master_ds,CMIP,GCM,ensemble,pathway,GWL=chosen_gwl)[var1]
gwl_rh = gwl.get_GWL_timeslice(hursmin_master_ds,CMIP,GCM,ensemble,pathway,GWL=chosen_gwl)[var2]
#gwl_rh = gwl_rh.assign_coords(time = pd.to_datetime(gwl_rh.time) + timedelta(hours = 12)) #needed

In [149]:
#gwl_rh.indexes['time'].to_datetimeindex()

#gwl_rh = gwl_rh.assign_coords(time = gwl_rh.indexes['time'].to_datetimeindex() + timedelta(hours = 12))
#gwl_rh['month'] = gwl_rh['time'].dt.month
#gwl_tasmax['month'] = gwl_tasmax['time'].dt.month

In [150]:
#Check that gwl_tasmax and gwl_rh have the same calendar
#gwl_tasmax.time.dt.calendar
#gwl_rh.time.dt.calendar

In [151]:
#Create the datasets for vpd and monthly mean vpd
gwl_vpd, monthly_mean_vpd = vpd_calc(gwl_rh, gwl_tasmax)

INFO:flox:Entering _validate_reindex: reindex is None
INFO:flox:Leaving _validate_reindex: method = None, returning None
INFO:flox:_choose_engine: Choosing 'numpy'
INFO:flox:find_group_cohorts: cohorts is preferred, chunking is perfect.
INFO:flox:_choose_method: method is None
INFO:flox:_choose_method: choosing preferred_method=cohorts
INFO:flox:Entering _validate_reindex: reindex is None
INFO:flox:Leaving _validate_reindex: reindex is False


In [152]:
#monthly_mean_vpd

In [153]:
#print vpd to an external file

file_name_vpd = pathway + '_' + GCM + '_' + RCM + '_gwl' + chosen_gwl + '_vpd.nc' 
output_file_location = output_dir + GCM + '/' + pathway + '/' + ensemble + '/' + RCM + '/v1-r1/day/' + file_name_vpd
gwl_vpd.to_netcdf(output_file_location, engine='netcdf4')
print(output_file_location)

/g/data/ia39/ncra/bushfire/vpd/NorESM2-MM/ssp370/r1i1p1f1/BARPA-R/v1-r1/day/ssp370_NorESM2-MM_BARPA-R_gwl3.0_vpd.nc


In [154]:
#print monthly mean ds to external file

file_name_mean = 'gwl' + chosen_gwl + '_monthly_mean_vpd_' + GCM + '_' + RCM + '_' + pathway + '_' + ensemble + '.nc' 
output_file_location = output_dir_mm + file_name_mean
monthly_mean_vpd.to_netcdf(output_file_location, engine='netcdf4')
print(output_file_location)

/g/data/ia39/ncra/bushfire/vpd/monthly_mean/gwl3.0_monthly_mean_vpd_NorESM2-MM_BARPA-R_ssp370_r1i1p1f1.nc
